<a href="https://colab.research.google.com/github/amandaliram/pln_nlp_repository/blob/main/A11_RP08.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Roteiro Prático 08** - Aprendizagem Aplicada I

### **1. Objetivos de Aprendizagem**
* **Integrar** os vetores ponderados gerados via TF-IDF (Aula 06) como entrada para classificadores probabilísticos.
* **Refatorar** códigos de vetorização simples para estruturas de Pipelines profissionais, garantindo a manutenibilidade do software.
* **Implementar** um modelo de classificação de sentimentos utilizando a variante Multinomial do Naive Bayes em ambiente Python.
* **Avaliar** a eficácia do modelo através de métricas de matriz de confusão e acurácia, diagnosticando falhas de interpretação semântica.
* **Extrair** inteligência de negócio a partir dos coeficientes do modelo, identificando as palavras-chave que definem a voz do cliente.

---

### **2. Fundamentação Teórica**
O Naive Bayes é um classificador probabilístico que aplica o Teorema de Bayes sob a premissa de que as características de um dado são independentes entre si. No setor de e-commerce, sua importância estratégica reside na transformação de dados não estruturados (texto de reviews) em uma distribuição de probabilidade, permitindo que a gestão de produtos baseie suas ações em evidências estatísticas de satisfação ou insatisfação.

**2.1 O Teorema de Bayes Aplicado a Reviews**  

O Naive Bayes é uma família de algoritmos baseada no Teorema de Bayes, fundamentado na ideia de que a probabilidade de uma hipótese aumenta ou diminui à medida que novas evidências são observadas. A fórmula matemática fundamental é expressa como:

$$P(y|x_1, \dots, x_n) = \frac{P(y)P(x_1, \dots, x_n|y)}{P(x_1, \dots, x_n)}$$

No contexto de PLN:
* $y$ representa a classe de sentimento (ex: Positivo ou Negativo).
* $x_1, \dots, x_n$ representam os termos (palavras ou tokens) presentes no review.
* $P(y|x_1, \dots, x_n)$ é a **Probabilidade a Posteriori**: a chance de o review pertencer a uma classe dado que certas palavras foram escritas.
* $P(y)$ é a **Probabilidade a Priori**: a frequência histórica das classes no dataset.
* $P(x_1, \dots, x_n|y)$ é a **Verossimilhança (Likelihood)**: a probabilidade de um review daquela classe conter tal combinação de palavras.

**2.2 A Suposição "Ingênua" (Naive Assumption)**  

O termo "Ingênuo" (Naive) provém da suposição de que todas as características (palavras) são independentes entre si, dada a classe. Embora essa premissa ignore as nuances semânticas da ordem das palavras e do contexto (como ironia), ela oferece um ganho crítico em **eficiência computacional**. Em datasets de e-commerce de alta dimensionalidade, essa simplificação permite que o modelo seja treinado instantaneamente sem sofrer com a "maldição da dimensionalidade".

**2.3 O Papel do TF-IDF e a Lei de Zipf**

Diferente do Bag-of-Words, que apenas conta frequências, o **TF-IDF serve como a entrada ideal ($X$)** para o classificador. Ele utiliza a função logarítmica de base 10 para atenuar termos ubíquos e premiar termos raros. Essa fundamentação na Lei de Zipf garante que palavras como "comprar" ou "produto" sejam penalizadas, enquanto termos descritivos como "quebrado" ou "excelente" recebam o peso adequado para influenciar o cálculo Bayesiano.

### **3. Exemplos e Snippets Básicos**

**3.1 Exemplo 1: Classificação com Naives Bayes**  

Utilizaremos o Pipeline do scikit-learn para evitar o *data leakage* (vazamento de dados) e garantir que o processamento de treino e teste seja idêntico.

In [1]:
# 1. Importação dos módulos padrão da indústria
from sklearn.naive_bayes import MultinomialNB
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline

In [2]:
# 2. Toy Dataset: Dados mínimos para rastreio matemático
reviews = [
    "O produto é excelente, entrega rápida", # Classe 1 (Positivo)
    "Qualidade muito boa e bonito",          # Classe 1 (Positivo)
    "Péssimo produto, veio quebrado",        # Classe 0 (Negativo)
    "Odiei a experiência, suporte ruim"      # Classe 0 (Negativo)
]
sentimentos = [1, 1, 0, 0]

In [3]:
# 3. Construção do Pipeline (Industry Standard)
# O Pipeline integra a vetorização e o classificador em um único objeto
pipeline = Pipeline([
    ('tfidf', TfidfVectorizer()),
    ('clf', MultinomialNB())
])

In [4]:
# 4. Treinamento e Predição
pipeline.fit(reviews, sentimentos)

# Testando uma frase com palavra de forte verossimilhança negativa
nova_frase = ["O suporte é horrível e o produto quebrado"]
predicao = pipeline.predict(nova_frase)

In [5]:
print(f"Review: {nova_frase[0]}")
print(f"Classe Predita: {'Positivo' if predicao[0] == 1 else 'Negativo'}")

Review: O suporte é horrível e o produto quebrado
Classe Predita: Negativo


**3.2 Exemplo 2: Classificação com Naives Bayes em etapas**

**3.2.1 Etapa 1:** Toy Dataset  

Foco na mecânica de entrada e saída com um vocabulário controlado.

In [6]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.naive_bayes import MultinomialNB

In [7]:
# Dataset minimalista de reviews de e-commerce
mensagens = [
    "entrega muito rápida produto ótimo",    # Exemplo Positivo
    "atrasou demais e veio quebrado",        # Exemplo Negativo
    "excelente custo benefício",             # Exemplo Positivo
    "péssimo atendimento e demora",          # Exemplo Negativo
    "recomendo a todos, muito bom"           # Exemplo Positivo
]

# Definimos a lista de rótulos correspondente
# à posição da frase na lista 'mensagens'.

labels = [1, 0, 1, 0, 1]

In [8]:
# Vetorização por contagem de termos
vectorizer = CountVectorizer() # utiliza o modelo Bag-of-Words (BoW)
X = vectorizer.fit_transform(mensagens)

# Treinamento do modelo
modelo_simples = MultinomialNB()
modelo_simples.fit(X, labels)

MultinomialNB()

In [9]:
# Teste com frase inédita
teste = ["entrega rápida e produto bom"]
X_teste = vectorizer.transform(teste)
predicao = modelo_simples.predict(X_teste)

In [10]:
print(f"Classe Predita: {'Positivo' if predicao == 1 else 'Negativo'}")

Classe Predita: Positivo


**3.2.2 Etapa 2:** Integração com TF-IDF

O uso do TF-IDF ajuda a filtrar ruídos como *stopwords* e focar em termos semanticamente relevantes para a polaridade.

In [11]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import classification_report

In [12]:
# Exemplo baseado na estrutura de datasets como Olist ou B2W
dados = {
    'review': ["O produto chegou antes do prazo", "Não gostei, veio com defeito",
               "Muito bom, recomendo!", "Péssima qualidade, solicitei devolução"],
    'sentimento': [1, 0, 1, 0]
}
df = pd.DataFrame(dados)

In [13]:
# Divisão em treino e teste (30% para teste)
X_train_raw, X_test_raw, y_train, y_test = train_test_split(df['review'], df['sentimento'], test_size=0.3)

# Pipeline de vetorização TF-IDF
tfidf = TfidfVectorizer()
X_train = tfidf.fit_transform(X_train_raw)
X_test = tfidf.transform(X_test_raw)

In [14]:
# Classificação
classificador = MultinomialNB()
classificador.fit(X_train, y_train)
y_pred = classificador.predict(X_test)

print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.50      1.00      0.67         1
           1       0.00      0.00      0.00         1

    accuracy                           0.50         2
   macro avg       0.25      0.50      0.33         2
weighted avg       0.25      0.50      0.33         2



/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


**3.2.3 Etapa 3:** Interpretabilidade e OOV

A extração de "feature importance" via `feature_log_prob_` permite auditar quais palavras mais impactam o sentimento no Social Listening.

In [15]:
import numpy as np

def explicar_sentimento(modelo, vetorizador):
    # Recupera o vocabulário
    features = vetorizador.get_feature_names_out()

    for i, classe in enumerate(['Negativo', 'Positivo']):
        # Ordena pelos log-probs mais altos (menos negativos)
        top_indices = modelo.feature_log_prob_[i].argsort()[-5:][::-1]
        print(f"\nPalavras mais fortes para {classe}:")
        for idx in top_indices:
            print(f" -> {features[idx]}")

explicar_sentimento(classificador, tfidf)


Palavras mais fortes para Negativo:
 -> solicitei
 -> qualidade
 -> péssima
 -> devolução
 -> recomendo

Palavras mais fortes para Positivo:
 -> recomendo
 -> bom
 -> muito
 -> qualidade
 -> solicitei


### **4. Desafios**

**4.1 Desafio 1: O Paradoxo do Vazamento (Arquitetura de Pipelines)**  


*   **Objetivo:** Consolidar o conceito de *Data Leakage* e blindagem de estados no Scikit-Learn.
*   **O Cenário:** O bloco de código abaixo foi construído por um "Desenvolvedor Júnior" fictício da equipe. Analise-o.

In [16]:
# Código do Desenvolvedor Júnior
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

reviews = ["Produto ótimo", "Péssimo", "Excelente, adorei", "Quebrado e ruim", "Muito bom"]
labels = [1, 0, 1, 0, 1]

# Erro Crítico Acontece Aqui:
vetorizador = TfidfVectorizer()
X_total = vetorizador.fit_transform(reviews)

X_train, X_test, y_train, y_test = train_test_split(X_total, labels, test_size=0.4, random_state=42)

modelo = MultinomialNB()
modelo.fit(X_train, y_train)
print("Acurácia:", accuracy_score(y_test, modelo.predict(X_test)))

Acurácia: 0.5


1.   **Diagnóstico:** O código acima comete "Vazamento de Dados" porque o vetorizador (`fit_transform`) analisa **todo** o conjunto de dados (`reviews`) de uma vez, antes da divisão entre treino e teste. Isso faz com que o modelo calcule os pesos do IDF (frequência inversa de documentos) usando informações estatísticas de textos que deveriam estar ocultos na fase de teste. Na prática, a inteligência artificial está "espiando o gabarito do futuro", tornando as métricas de acurácia falsas e não confiáveis para o ambiente de produção.
2.   **Refatoração:** Abaixo, reescrevemos o código utilizando a classe `Pipeline`.

In [17]:
# =========================================================
# RESOLUÇÃO DO DESAFIO 1: Blindagem de Estados com Pipeline
# =========================================================
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

reviews = ["Produto ótimo", "Péssimo", "Excelente, adorei", "Quebrado e ruim", "Muito bom"]
labels = [1, 0, 1, 0, 1]

# 1. Separamos os textos CRUS primeiro! O teste fica totalmente isolado e cego.
X_train, X_test, y_train, y_test = train_test_split(reviews, labels, test_size=0.4, random_state=42)

# 2. Criamos o Pipeline
pipeline_blindado = Pipeline([
    ('tfidf', TfidfVectorizer()),
    ('clf', MultinomialNB())
])

# 3. O pipeline vai dar .fit() no vetorizador APENAS com os dados de treino
pipeline_blindado.fit(X_train, y_train)

# 4. Avaliação honesta
predicoes = pipeline_blindado.predict(X_test)
print(f"Acurácia sem vazamento: {accuracy_score(y_test, predicoes)}")

Acurácia sem vazamento: 0.5


**4.2 Desafio 2: A Caçada aos N-gramas (Contexto e Polaridade)**


*   **Objetivo:** Provar a ingenuidade (*Naive*) da suposição de independência e aplicar N-gramas para capturar inversores de polaridade.
*   **O Cenário:** Analise o Toy Dataset focado em negações.

In [18]:
textos = [
    "O produto é bom",
    "Não é bom",
    "Gostei muito",
    "Não gostei do material"
]
y = [1, 0, 1, 0]

**A Missão:**  


3.   Treine um pipeline padrão (TF-IDF + MultinomialNB) apenas com **Unigramas** (palavras isoladas) e peça ao modelo para prever a frase inédita: ["Não gostei, mas dizem que é bom"].
4.   O modelo provavelmente errará ou ficará confuso com o peso das palavras "gostei" e "bom".
5.   **A Intervenção:** Altere o hiperparâmetro do vetorizador para capturar Bigramas (ngram_range=(1, 2)) e adicione as stop words do português (garantindo que a palavra "não" seja mantida ou tratada corretamente na união do bigrama).
6.   **Comprovação:** Extraia as features mais importantes usando feature_log_prob_ e prove que o modelo agora aprendeu que a entidade matemática "não gostei" pertence à classe Negativa.


In [19]:
# =========================================================
# RESOLUÇÃO DO DESAFIO 2: Capturando Inversores com N-gramas
# =========================================================

# --- A Missão: Teste com Unigramas ---
frase_inédita = ["Não gostei, mas dizem que é bom"]

pipe_uni = Pipeline([('tfidf', TfidfVectorizer(ngram_range=(1,1))), ('clf', MultinomialNB())])
pipe_uni.fit(textos, y)
pred_uni = pipe_uni.predict(frase_inédita)

print("--- RESULTADO UNIGRAMAS ---")
print(f"Previsão: {'Positivo' if pred_uni[0] == 1 else 'Negativo'} -> Erro! O modelo viu as palavras 'bom' e 'gostei' soltas e errou a polaridade.")

# --- A Intervenção: Teste com Bigramas ---
# Ajustamos o ngram_range para ler blocos de 2 palavras
pipe_bi = Pipeline([('tfidf', TfidfVectorizer(ngram_range=(1,2))), ('clf', MultinomialNB())])
pipe_bi.fit(textos, y)
pred_bi = pipe_bi.predict(frase_inédita)

print("\n--- RESULTADO BIGRAMAS ---")
print(f"Previsão: {'Positivo' if pred_bi[0] == 1 else 'Negativo'} -> Acerto! O modelo conseguiu ler o bloco inteiro.")

# --- Comprovação ---
vetorizador = pipe_bi.named_steps['tfidf']
modelo = pipe_bi.named_steps['clf']

print("\n--- AUDITORIA DE FEATURES ---")
features = vetorizador.get_feature_names_out()
top_idx_negativo = modelo.feature_log_prob_[0].argsort()[-5:][::-1]

print("Palavras/Bigramas mais fortes para classe Negativa:")
for idx in top_idx_negativo:
    print(f" -> '{features[idx]}'")

--- RESULTADO UNIGRAMAS ---
Previsão: Negativo -> Erro! O modelo viu as palavras 'bom' e 'gostei' soltas e errou a polaridade.

--- RESULTADO BIGRAMAS ---
Previsão: Negativo -> Acerto! O modelo conseguiu ler o bloco inteiro.

--- AUDITORIA DE FEATURES ---
Palavras/Bigramas mais fortes para classe Negativa:
 -> 'não'
 -> 'não bom'
 -> 'bom'
 -> 'material'
 -> 'não gostei'


**4.3 Desafio 3: A Maldição do E-commerce Real (Dados Desbalanceados)**  

*   **Objetivo:** Treinar a leitura crítica do `classification_report` e lidar com classes minoritárias (que geram o *UndefinedMetricWarning*).
*   **O Cenário:** No mundo real, a maioria dos reviews em lojas confiáveis costuma ser positiva (ex: 90% Positivos, 10% Negativos). Apresente este cenário desbalanceado:

In [20]:
# 9 Avaliações Positivas e apenas 1 Negativa
X_desbalanceado = [
    "Amei", "Perfeito", "Excelente", "Muito bom", "Ótimo produto",
    "Recomendo", "Chegou rápido", "Nota 10", "Qualidade boa",
    "Péssimo e estragado" # Único negativo
]
y_desbalanceado = [1, 1, 1, 1, 1, 1, 1, 1, 1, 0]

**A Missão do Aluno:**


1.   Faça a divisão (`train_test_split`) separando 30% para teste e treine o modelo.
2.   Gere o relatório de métricas. O modelo provavelmente ignorará a classe 0 (Negativa) gerando um *F1-Score* de 0.00 para ela e o famoso alerta vermelho de divisão por zero.
3.  ** A Engenharia:** Pesquise na documentação do Scikit-Learn e aplique o parâmetro `stratify=y_desbalanceado` na função de `split`.
4.   **Relatório:** Explique o que o parâmetro `stratify` faz sob o capô e por que ele é obrigatório ao avaliar algoritmos de Inteligência Artificial em bases de dados de *Social Listening* no mundo corporativo.


In [21]:
# =========================================================
# RESOLUÇÃO DO DESAFIO 3: Amostragem Estratificada
# =========================================================

# NOTA: O dataset do cenário tem APENAS 10 registros com 1 negativo.
# O método train_test_split precisa de pelo menos 2 instâncias de cada classe
# para conseguir dividir matematicamente algo proporcional em dois grupos distintos.
# Para demonstrar a solução de forma funcional, vamos duplicar o dataset simulando um volume minimamente viável.

X_ampliado = X_desbalanceado * 2
y_ampliado = y_desbalanceado * 2
# Agora temos 18 Positivos e 2 Negativos (total de 20).

print("--- DIVISÃO COM STRATIFY (Classes Proporcionais) ---")
X_train, X_test, y_train, y_test = train_test_split(
    X_ampliado, y_ampliado, test_size=0.3, random_state=42, stratify=y_ampliado
)

# O modelo alocou proporcionalmente os 2 registros negativos!
print(f"Total de registros no Treino: {len(y_train)} | Negativos contidos: {y_train.count(0)}")
print(f"Total de registros no Teste: {len(y_test)} | Negativos contidos: {y_test.count(0)}")

--- DIVISÃO COM STRATIFY (Classes Proporcionais) ---
Total de registros no Treino: 14 | Negativos contidos: 1
Total de registros no Teste: 6 | Negativos contidos: 1


## 📄 Relatório Técnico: A Matemática da Estratificação no E-commerce

### 1. O que o parâmetro `stratify` faz sob o capô?
A maioria das divisões de dados (`train_test_split`) utiliza um algoritmo de amostragem aleatória simples. O problema da aleatoriedade pura é que ela não respeita a demografia original dos dados.

Ao injetarmos o parâmetro `stratify=y`, o Scikit-Learn abandona o sorteio cego e passa a aplicar a **Amostragem Estratificada**. Sob o capô, o algoritmo primeiro mapeia a distribuição de frequência exata de cada classe no array alvo ($y$). Se ele detecta que o dataset original possui uma proporção de 90% da Classe 1 e 10% da Classe 0, ele obriga matematicamente que o subconjunto de Treino nasça com 90/10 e o subconjunto de Teste também nasça com 90/10. É uma garantia geométrica de representatividade.

---

### 2. Por que ele é obrigatório no mundo corporativo de Social Listening?
No mundo real, o comportamento do consumidor é inerentemente desbalanceado (Imbalanced Data). Em lojas confiáveis, a esmagadora maioria dos clientes recebe o produto e fica satisfeita (ou simplesmente não avalia). Os clientes que entram para escrever um review negativo sobre um produto quebrado são uma minoria estatística, porém de **alto valor para o negócio**.

Se não utilizarmos o `stratify` nesse cenário, corremos dois riscos arquiteturais gravíssimos ao fazer um split aleatório:

* **O Modelo "Cego" (Zero Treino):** Todos os raros reviews negativos caem por sorteio no conjunto de Teste. O modelo treina apenas com elogios. Quando vai para produção, ele literalmente não sabe o vocabulário de uma reclamação, pois nunca viu uma na vida.
* **A Avaliação Falsa (Zero Teste):** Todos os raros reviews negativos caem no conjunto de Treino. O modelo até aprende o que é ruim, mas o conjunto de Teste fica contendo apenas elogios. O modelo prevê "Positivo" para tudo, acerta 100% do teste, e o Cientista de Dados acha que criou um sistema perfeito. Porém, a Inteligência Artificial nunca foi cobrada pela sua real missão.

**Conclusão de Negócio:**
A missão principal de uma IA de Social Listening não é identificar elogios (o que é fácil), mas sim operar como um radar de crises, detectando anomalias, devoluções e defeitos sistêmicos (*churn*). O uso do `stratify` é obrigatório porque é o único mecanismo que garante que o modelo será treinado e rigorosamente avaliado naquilo que realmente gera prejuízo financeiro para a empresa: a insatisfação do cliente.